# Avian Interaction Extraction - Structured Output

In [1]:
import os 
import requests
import json
from pathlib import Path
from IPython.display import Markdown, display

import pandas as pd

from src.util import show_table, show_response_stats, LLMOptions

OLLAMA_HOST = "127.0.0.1:40063"
os.environ["OLLAMA_HOST"] = "127.0.0.1:40063"
print(OLLAMA_HOST)

LLM_MODEL = "llama3.2:3b"

data_dir = Path('/mnt/ufs18/rs-037/jdevenegas_research/NSF-DBI/data')

127.0.0.1:40063


## Making a basic Ollama chat api wrapper 
We define a simple function that will allow us to send chat-style prompts to the local LLM server and returns the full response JSON, optionally including system context and example chat pairs (one-shot or few-shot prompting). 

The `/api/generate` endpoint is used primarily for one-off tasks, or single self-contained questions/queries. The `/api/chat` endpoint is primarily used for back-and-forth chatting, with the ability to preserve and manage chat history through the `messages` array. 

> Ollama’s /api/chat endpoint is stateless by default, and must be managed through `messages`

Here are some of the response items, with their definitions. See [Ollama /api/chat](https://docs.ollama.com/api/chat) for more information. 

`prompt_eval_count`: Number of tokens in the prompt

`eval_count`: Number of tokens generated in the response

`total_duration`: Total time spent generating in nanoseconds

`prompt_eval_duration`: Time spent evaluating the prompt in nanoseconds

`eval_duration`: Time spent generating tokens in nanoseconds

In [2]:
def ollama_chat(user_prompt, system_prompt = None, example_pair = None, 
                llm_options=None, content_only=False):
    """send prompt to llm and return response"""

    if llm_options:
        print(f"runtime params: {llm_options}")
    
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    if example_pair:
        example_input, example_output = example_pair
        messages.append({"role": "user", "content": example_input})
        messages.append({"role": "assistant", "content": example_output})

    messages.append({"role": "user", "content": user_prompt})
        
    try:
        response = requests.post(
            f"http://{OLLAMA_HOST}/api/chat",
            json={
                "model": LLM_MODEL,
                "stream": False,
                "options": llm_options,
                "messages": messages
            }
        )
        data = response.json()
        if content_only:
            return data['message']['content']
        else: 
            return data
        
    except Exception as e:
        print(f"Error during llm call: {e}")
        return None

In this example below, you could use curl in the terminal to get a comprable response (here using generate endpoint)
```
curl http://localhost:40063/api/generate -d '{
  "model": "llama3.2:3b",
  "prompt": "Why is the sky blue?",
  "stream": false,
  "options": {
    "num_ctx": 2048,
    "temperature": 0.8
  }
}'
```

### setting runtime paramters
You can add control (runtime) parameters to guide model behaviour and output. These are added in the json payload in the request format, as seen in our `ollama_chat` function and in the curl example above. Ollama docs provide a [list of valid parameters](https://docs.ollama.com/modelfile#valid-parameters-and-values) with default values and descriptions, please reference this when testing different runtime parameters. In our setup, we've defined a schema with the `LLMOptions` class for handling these runtime parameters, to validate against the supported set of parameters. Note we've only set default values for `num_ctx` and `temperature` in this class, but all other parameters you leave unset will simply be set to default by Ollama when running the models. 

> Note: we set default `n_ctx` low to ensure our base model (llama3.2:3b) runs without memory errors on A100 slices (10gb vram). Without this set, it would default to the model (or Ollama's) default context length (~131072 here), which will often raise memory errors, esspecialy if other runtime options (e.g., `OLLAMA_KV_CACHE_TYPE' and `OLLAMA_FLASH_ATTENTION`) are not set to mitigate the larger memory. 

In [3]:
llm_opts = LLMOptions()
llm_opts_dict = llm_opts.model_dump(exclude_none=True)

response = ollama_chat("Why is the sky blue?", llm_options=llm_opts_dict)
print(response)

runtime params: {'num_ctx': 2048, 'temperature': 0.8}
{'model': 'llama3.2:3b', 'created_at': '2026-03-30T12:17:32.189382169Z', 'message': {'role': 'assistant', 'content': "The sky appears blue because of a phenomenon called scattering, which occurs when sunlight interacts with tiny molecules of gases in the Earth's atmosphere. Here's a simplified explanation:\n\n1. **Sunlight enters the Earth's atmosphere**: When sunlight enters the Earth's atmosphere, it consists of a spectrum of colors, including red, orange, yellow, green, blue, indigo, and violet.\n2. **Scattering by atmospheric molecules**: The shorter (blue) wavelengths of light are scattered more efficiently than the longer (red) wavelengths by the tiny molecules of gases in the atmosphere, such as nitrogen (N2) and oxygen (O2).\n3. **Blue light is dispersed**: As a result of this scattering, the blue light is dispersed in all directions, reaching our eyes from every part of the sky.\n4. **Red light continues straight**: The lon

You can use the `show_response_stats` helper function to display some basic stats (e.g., duration and token counts) in a table. 

In [4]:
show_response_stats(response)


| Metric | Description | Units | Value |
|--------|-------------|-------|-------|
| `prompt_eval_count` | Number of tokens in the prompt | tokens | 31 |
| `eval_count` | Number of tokens generated in the response | tokens | 364 |
| `total_duration` | Total time spent for the request | nanoseconds | 6606940423 |
| `prompt_eval_duration` | Time spent evaluating the prompt | nanoseconds | 21621953 |
| `eval_duration` | Time spent generating tokens | nanoseconds | 6108605558 |


You may add system context to the prompt through the `system_prompt` input. 

In [5]:
system_prompt = "You are an expert atmospheric physicist. Give concise answers, limiting your responses to 2-3 sentences." 
response = ollama_chat("Why is the sky blue?", system_prompt=system_prompt, llm_options=llm_opts_dict)
print(response)

show_response_stats(response)

runtime params: {'num_ctx': 2048, 'temperature': 0.8}
{'model': 'llama3.2:3b', 'created_at': '2026-03-30T12:17:36.4492422Z', 'message': {'role': 'assistant', 'content': "The sky appears blue due to a phenomenon called Rayleigh scattering, where shorter (blue) wavelengths of sunlight scatter more than longer (red) wavelengths as they pass through the Earth's atmosphere. This scattering effect favors the blue end of the visible spectrum, making it the dominant color we see. The amount of scattering decreases with increasing wavelength, resulting in the red and orange hues appearing less prominent at higher altitudes."}, 'done': True, 'done_reason': 'stop', 'total_duration': 1668405275, 'load_duration': 181652618, 'prompt_eval_count': 52, 'prompt_eval_duration': 45262566, 'eval_count': 83, 'eval_duration': 1380469340}



| Metric | Description | Units | Value |
|--------|-------------|-------|-------|
| `prompt_eval_count` | Number of tokens in the prompt | tokens | 52 |
| `eval_count` | Number of tokens generated in the response | tokens | 83 |
| `total_duration` | Total time spent for the request | nanoseconds | 1668405275 |
| `prompt_eval_duration` | Time spent evaluating the prompt | nanoseconds | 45262566 |
| `eval_duration` | Time spent generating tokens | nanoseconds | 1380469340 |


If you'd like to only get the message content of the response, you can parse it out yourself using the 'message' and 'conent' keys, or just use the `content_only` option (set to True) in the `ollama_chat` function. 

In [6]:
response = ollama_chat("Why is the sky blue?", system_prompt=system_prompt, llm_options=llm_opts_dict,
                      content_only=True)

print(response)

runtime params: {'num_ctx': 2048, 'temperature': 0.8}
The sky appears blue due to a phenomenon called Rayleigh scattering, where shorter (blue) wavelengths of light scatter more than longer (red) wavelengths when interacting with tiny molecules in the atmosphere. This scattering effect favors shorter wavelengths, resulting in the blue color we see during the daytime. The color is also influenced by atmospheric conditions and pollution levels.


## Interaction Extraction
We now will work on extracting information in a structured format. The use-case here is extracting species interactions from avian interaction text excerpts. First, let's look at an example input-output pair we'll use.

In [7]:
#load system prompt and user-assistant example pair
with open(data_dir / "system.txt", "r") as f:
    system_prompt = f.read().strip()
    
with open(data_dir / "example_input.txt", "r") as f:
    example_input = f.read().strip()

with open(data_dir / "example_output.txt", "r") as f:
    example_output = f.read().strip()


print("Example input: ", example_input)
print("\n\nExample output:\n" + example_output)

Example input:  Actor species (species 1) common name: Cooper's Hawk, scientific name: Astur cooperii, genus combo name: Astur cooperii. Major Food Items: See Table 2. Relative importance of birds versus mammals varies across regions and among habitat types. Almost all data from breeding season, mostly from southeastern, midwestern and southwestern United States, and southwestern British Columbia. Of 12 studies across North America, 10 (83%) reported bird prey to be dominant, ranging from 56% to 96% of total items tallied (see summary in 1). The most common from smaller to larger: House Sparrow (example: ML81132121) (especially in cities such as Victoria, British Columbia; Milwaukee, Wisconsin; Grand Forks, North Dakota; and Albuquerque, New Mexico; 50, RNR); the Northern Cardinal (Cardinalis cardinalis) and the Northern Mockingbird (Mimus polyglottos); especially in rural Florida; 46); and the American Robin (example: ML40321891), European Starling (example: ML51466291), Blue Jay (Cya

In [8]:
show_table(example_output)

| species1_common | species1_scientific | genus_species_combo_sp1 | species2_common | species2_scientific | genus_species_combo_sp2 | label(type_of_interaction) | effect_on_sp1 | effect_on_sp2 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Cooper's Hawk | Astur cooperii | Astur cooperii | (unid. Red / Yellow Shafted) Northern Flicker | Colaptes auratus | Colaptes auratus auratus | predation | 1 | -1 |
| Cooper's Hawk | Astur cooperii | Astur cooperii | Eurasian Collared-Dove | Streptopelia decaocto | Streptopelia decaocto | predation | 1 | -1 |

Now let's use the example input for our user input, to see the extracted interaction table. Note in this specific response, it returned a nearly identical table as the example ouput. However, this will vary with each run, so you may see different results (e.g., more or less interactions). You'll see here this system context is very verbose and is a brute force approach to get the desired behaviour and output we want. To accomodate this, we'll need to expand the context length significantly. 

In [9]:
llm_opts = LLMOptions(num_ctx=20000)
llm_opts_dict = llm_opts.model_dump(exclude_none=True)

response = ollama_chat(example_input, system_prompt=system_prompt, 
                       example_pair=(example_input, example_output), 
                      llm_options=llm_opts_dict)

show_response_stats(response)

runtime params: {'num_ctx': 20000, 'temperature': 0.8}



| Metric | Description | Units | Value |
|--------|-------------|-------|-------|
| `prompt_eval_count` | Number of tokens in the prompt | tokens | 18515 |
| `eval_count` | Number of tokens generated in the response | tokens | 148 |
| `total_duration` | Total time spent for the request | nanoseconds | 23441966862 |
| `prompt_eval_duration` | Time spent evaluating the prompt | nanoseconds | 16023163632 |
| `eval_duration` | Time spent generating tokens | nanoseconds | 4143082266 |


In [10]:
out = response['message']['content']
print(out)

species1_common|species1_scientific|genus_species_combo_sp1|species2_common|species2_scientific|genus_species_combo_sp2|label(type_of_interaction)|effect_on_sp1|effect_on_sp2
Cooper's Hawk|Astur cooperii|Astur cooperii|(unid. Red / Yellow Shafted) Northern Flicker|Colaptes auratus|Colaptes auratus auratus|predation|-1|1 
Cooper's Hawk|Astur cooperii|Astur cooperii|Eurasian Collared-Dove|Streptopelia decaocto|Streptopelia decaocto|predation|-1|1


In [11]:
show_table(out)

| species1_common | species1_scientific | genus_species_combo_sp1 | species2_common | species2_scientific | genus_species_combo_sp2 | label(type_of_interaction) | effect_on_sp1 | effect_on_sp2 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Cooper's Hawk | Astur cooperii | Astur cooperii | (unid. Red / Yellow Shafted) Northern Flicker | Colaptes auratus | Colaptes auratus auratus | predation | -1 | 1  |
| Cooper's Hawk | Astur cooperii | Astur cooperii | Eurasian Collared-Dove | Streptopelia decaocto | Streptopelia decaocto | predation | -1 | 1 |

We'll now look at a single text excerpt that is not included in the example pair.

In [12]:
with open(data_dir / f"avian_interaction_docs/para_{1}.txt", "r") as f:
    user_prompt = f.read().strip()
print(user_prompt)

Actor species (species 1) common name: American Goshawk, scientific name: Astur atricapillus, genus combo name: Astur atricapillus. Major Food Items: Southern populations may depend less on mammalian prey than northern populations, but there are exceptions (Table 2; Boal and Mannan 1994). In boreal regions, grouse are fundamental foods, accounting for up to 80% of prey, particularly for chicks and juveniles. Although potential prey species are extensive (Appendix 1), a few taxa are particularly prevalent in most diets. Dominant mammalian prey in North America include tree squirrels (Douglas squirrel [Tamiasciurus douglasii], red squirrel [T. hudsonicus], gray squirrel [Sciurus carolinensis], tassel-eared squirrel [S. aberti], northern flying squirrel [Glaucomys sabrinus]); ground squirrels (golden-mantled ground squirrel [Spermophilus lateralis], Richardson's ground squirrel [S. richardsonii], Belding's ground squirrel [S. beldingi], Townsend's ground squirrel [S. townsendii]); and lag

In [13]:
response = ollama_chat(user_prompt, system_prompt=system_prompt, 
                       example_pair=(example_input, example_output), 
                      llm_options=llm_opts_dict)

show_response_stats(response)

out = response['message']['content']
print(out)

runtime params: {'num_ctx': 20000, 'temperature': 0.8}



| Metric | Description | Units | Value |
|--------|-------------|-------|-------|
| `prompt_eval_count` | Number of tokens in the prompt | tokens | 18831 |
| `eval_count` | Number of tokens generated in the response | tokens | 99 |
| `total_duration` | Total time spent for the request | nanoseconds | 4061531885 |
| `prompt_eval_duration` | Time spent evaluating the prompt | nanoseconds | 944848608 |
| `eval_duration` | Time spent generating tokens | nanoseconds | 2775510932 |


species1_common|species1_scientific|genus_species_combo_sp1|species2_common|species2_scientific|genus_species_combo_sp2|label(type_of_interaction)|effect_on_sp1|effect_on_sp2
American Goshawk|Astur atricapillus|Astur atricapillus|(unid. Red / Yellow Shafted) Northern Flicker|Colaptes auratus|Colaptes auratus auratus|predation|1|-1


In [14]:
show_table(out)

| species1_common | species1_scientific | genus_species_combo_sp1 | species2_common | species2_scientific | genus_species_combo_sp2 | label(type_of_interaction) | effect_on_sp1 | effect_on_sp2 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| American Goshawk | Astur atricapillus | Astur atricapillus | (unid. Red / Yellow Shafted) Northern Flicker | Colaptes auratus | Colaptes auratus auratus | predation | 1 | -1 |

### Adding structured output format
Instead of specifying structured output / format in the system prompt, we can add the format as a JSON schema in the `/api/chat` requests. Here we re-define `ollama_chat` to use the desired format for the extracted interactions table. This can be done by defining and adding a json schema in the format option of the json payload for the request. In this example we we'll hard code the json schema in the `ollama_chat_struct` function, but you can generalize this to allow for a json schema output format parameter in your chat wrapper. Of course, you could also define a single chat wrapper that handles both structured and unstructured response. 

In [29]:
def ollama_chat_struct(user_prompt, system_prompt = None, example_pair = None,
                      llm_options=None, content_only=False):
    """send prompt to llm and return structured response"""

    if llm_options:
        print(f"runtime params: {llm_options}")

    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    if example_pair:
        example_input, example_output = example_pair
        messages.append({"role": "user", "content": example_input})
        messages.append({"role": "assistant", "content": example_output})

    messages.append({"role": "user", "content": user_prompt})

    # Define the structured output schema for species interactions
    output_format = {
        "type": "object",
        "properties": {
            "interactions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "species1_common": {"type": "string"},
                        "species1_scientific": {"type": "string"},
                        "genus_species_combo_sp1": {"type": "string"},
                        "species2_common": {"type": "string"},
                        "species2_scientific": {"type": "string"},
                        "genus_species_combo_sp2": {"type": "string"},
                        "label": {"type": "string"},
                        "effect_on_sp1": {"type": "integer"},
                        "effect_on_sp2": {"type": "integer"}
                    },
                    "required": [
                        "species1_common",
                        "species1_scientific",
                        "genus_species_combo_sp1",
                        "species2_common",
                        "species2_scientific",
                        "genus_species_combo_sp2",
                        "label",
                        "effect_on_sp1",
                        "effect_on_sp2"
                    ]
                }
            }
        },
        "required": ["interactions"]
    }
    
    try:
        response = requests.post(
            f"http://{OLLAMA_HOST}/api/chat",
            json={
                "model": LLM_MODEL,
                "stream": False,
                "options": llm_options,
                "messages": messages,
                "format": output_format  # add structured output
            }
        )
        data = response.json()
        if content_only:
            return data['message']['content']
        else: 
            return data
        
    except Exception as e:
        print(f"Error during llm call: {e}")
        return None

We can now remove the explicit output format definitions (pipe delimited table) from the system context, and rely on the output format we defined in the structured output chat wrapper. 

In [16]:
with open(data_dir / "system-nofmt.txt", "r") as f:
    system_prompt_nofmt = f.read().strip()

response = ollama_chat_struct(user_prompt, system_prompt=system_prompt_nofmt, 
                              example_pair=(example_input, example_output), 
                              llm_options=llm_opts_dict)

show_response_stats(response)

out = response['message']['content']
print(out)


| Metric | Description | Units | Value |
|--------|-------------|-------|-------|
| `prompt_eval_count` | Number of tokens in the prompt | tokens | 18707 |
| `eval_count` | Number of tokens generated in the response | tokens | 545 |
| `total_duration` | Total time spent for the request | nanoseconds | 41229387493 |
| `prompt_eval_duration` | Time spent evaluating the prompt | nanoseconds | 16342355325 |
| `eval_duration` | Time spent generating tokens | nanoseconds | 15561028081 |


{

"interactions": [
{
 "species1_common": "American Goshawk",
 "species1_scientific": "Astur atricapillus",
 "genus_species_combo_sp1": "Astur atricapillus",
 "species2_common": "Northern Flicker",
 "species2_scientific": "Colaptes auratus",
 "genus_species_combo_sp2": "Colaptes auratus",
 "label": "predation",
 "effect_on_sp1": 1,
 "effect_on_sp2": -1
},
{
 "species1_common": "American Goshawk",
 "species1_scientific": "Astur atricapillus",
 "genus_species_combo_sp1": "Astur atricapillus",
 "species2_common": "Northern Flicker",
 "species2_scientific": "Colaptes auratus",
 "genus_species_combo_sp2": "Colaptes auratus",
 "label": "predation",
 "effect_on_sp1": 1,
 "effect_on_sp2": -1
},
{
 "species1_common": "American Goshawk",
 "species1_scientific": "Astur atricapillus",
 "genus_species_combo_sp1": "Astur atricapillus",
 "species2_common": "Blue Jay",
 "species2_scientific": "Cyanocitta cristata",
 "genus_species_combo_sp2": "Cyanocitta cristata",
 "label": "predation",
 "effect_on_

In [18]:
out_json = json.loads(out)
df = pd.DataFrame(out_json.get("interactions", []))
display(df)

,species1_common,species1_scientific,genus_species_combo_sp1,species2_common,species2_scientific,genus_species_combo_sp2,label,effect_on_sp1,effect_on_sp2
0,American Goshawk,Astur atricapillus,Astur atricapillus,Northern Flicker,Colaptes auratus,Colaptes auratus,predation,1,-1
1,American Goshawk,Astur atricapillus,Astur atricapillus,Northern Flicker,Colaptes auratus,Colaptes auratus,predation,1,-1
2,American Goshawk,Astur atricapillus,Astur atricapillus,Blue Jay,Cyanocitta cristata,Cyanocitta cristata,predation,1,-1
3,American Goshawk,Astur atricapillus,Astur atricapillus,Steller's Jay,Cyanocitta stelleri,Cyanocitta stelleri,predation,1,-1
4,American Goshawk,Astur atricapillus,Astur atricapillus,American Crow,Corvus brachyrhynchos,Corvus brachyrhynchos,predation,1,-1


We can further reduce the system context by stripping down the rules and instructions we give the models. In this case, we may consider removing the large taxonomy table from the system context, and simply have the model output the species as they're named directly in the given text, rather than matching against the taxonomy table to add the different levels of naming. We could instead take the raw output and do additional lookup (w/ fuzzy matching) steps to populate those fields. This would be a natural step in an agentic approach, where we would get the raw output then have a tool or additional LLM work specifically on name matching and populating additional fields from taxonomy tables or other descriptors. 

In this example, we'll reduce the naming fields to only `species_1` and `species_2` with the system context directing the model to populate these with the names as written in the text. With these changes in the system context, you will see a significant reduction in the full prompt context length.

In [19]:
# Load example from file
with open(data_dir / "example_output_minimal.json", "r") as f:
    example_output_minimal = json.load(f)

with open(data_dir / "system_notaxa.txt", "r") as f:
    system_prompt_minimal = f.read().strip()

In [20]:
df = pd.DataFrame(example_output_minimal.get("interactions", []))
display(df)

,species_1,species_2,label(type_of_interaction),effect_on_sp1,effect_on_sp2
0,Cooper's Hawk,Northern Flicker (Colaptes auratus),predation,1,-1
1,Cooper's Hawk,Doves (Streptopelia decaocto),predation,1,-1


We'll need to re-define the output format accordingly in the chat wrapper. We can also update the example output in the example pair to follow this re-defined (structured) format. 

In [30]:
def ollama_chat_struct(user_prompt, system_prompt = None, example_pair = None,
                      llm_options=None, content_only=False):
    """send prompt to llm and return structured response"""

    if llm_options:
        print(f"runtime params: {llm_options}")

    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    if example_pair:
        example_input, example_output = example_pair
        messages.append({"role": "user", "content": example_input})
        messages.append({"role": "assistant", "content": json.dumps(example_output, indent=2)})

    messages.append({"role": "user", "content": user_prompt})

    # Define the structured output schema for species interactions
    output_format = {
        "type": "object",
        "properties": {
            "interactions": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "species_1": {"type": "string"},
                        "species_2": {"type": "string"},
                        "label(type_of_interaction)": {"type": "string"},
                        "effect_on_sp1": {"type": "integer"},
                        "effect_on_sp2": {"type": "integer"}
                    },
                    "required": [
                        "species_1",
                        "species_2",
                        "label(type_of_interaction)",
                        "effect_on_sp1",
                        "effect_on_sp2"
                    ]
                }
            }
        },
        "required": ["interactions"]
    }
    
    try:
        response = requests.post(
            f"http://{OLLAMA_HOST}/api/chat",
            json={
                "model": LLM_MODEL,
                "stream": False,
                "options": llm_options,
                "messages": messages,
                "format": output_format  # add structured output
            }
        )
        data = response.json()
        if content_only:
            return data['message']['content']
        else: 
            return data
        
    except Exception as e:
        print(f"Error during llm call: {e}")
        return None

In [31]:
llm_opts = LLMOptions(num_ctx=4096)
llm_opts_dict = llm_opts.model_dump(exclude_none=True)

response = ollama_chat_struct(user_prompt, system_prompt=system_prompt_minimal, 
                              example_pair=(example_input, example_output_minimal), 
                              llm_options=llm_opts_dict)

show_response_stats(response)

out = response['message']['content']
print(out)

runtime params: {'num_ctx': 4096, 'temperature': 0.8}



| Metric | Description | Units | Value |
|--------|-------------|-------|-------|
| `prompt_eval_count` | Number of tokens in the prompt | tokens | 1835 |
| `eval_count` | Number of tokens generated in the response | tokens | 1662 |
| `total_duration` | Total time spent for the request | nanoseconds | 57692915906 |
| `prompt_eval_duration` | Time spent evaluating the prompt | nanoseconds | 20961125 |
| `eval_duration` | Time spent generating tokens | nanoseconds | 30501015801 |


{
  "interactions": [
    {
      "species_1": "American Goshawk",
      "species_2": "Douglas squirrel (Tamiasciurus douglasii)",
      "label(type_of_interaction)": "predation",
      "effect_on_sp1": 1,
      "effect_on_sp2": -1
    },
    {
      "species_1": "American Goshawk",
      "species_2": "red squirrel (T. hudsonicus)",
      "label(type_of_interaction)": "predation",
      "effect_on_sp1": 1,
      "effect_on_sp2": -1
    },
    {
      "species_1": "American Goshawk",
      "species_2": "gray squirrel (Sciurus carolinensis)",
      "label(type_of_interaction)": "predation",
      "effect_on_sp1": 1,
      "effect_on_sp2": -1
    },
    {
      "species_1": "American Goshawk",
      "species_2": "tassel-eared squirrel (S. aberti)",
      "label(type_of_interaction)": "predation",
      "effect_on_sp1": 1,
      "effect_on_sp2": -1
    },
    {
      "species_1": "American Goshawk",
      "species_2": "northern flying squirrel (Glaucomys sabrinus)",
      "label(type_of_in

In [32]:
out_json = json.loads(out)
df = pd.DataFrame(out_json.get("interactions", []))
display(df)

,species_1,species_2,label(type_of_interaction),effect_on_sp1,effect_on_sp2
0,American Goshawk,Douglas squirrel (Tamiasciurus douglasii),predation,1,-1
1,American Goshawk,red squirrel (T. hudsonicus),predation,1,-1
2,American Goshawk,gray squirrel (Sciurus carolinensis),predation,1,-1
3,American Goshawk,tassel-eared squirrel (S. aberti),predation,1,-1
4,American Goshawk,northern flying squirrel (Glaucomys sabrinus),predation,1,-1
5,American Goshawk,golden-mantled ground squirrel (Spermophilus l...,predation,1,-1
6,American Goshawk,Richardson's ground squirrel (S. richardsonii),predation,1,-1
7,American Goshawk,Belding's ground squirrel (S. beldingi),predation,1,-1
8,American Goshawk,Townsend's ground squirrel (S. townsendii),predation,1,-1
9,American Goshawk,cottontails (Sylvilagus sp.),predation,1,-1


### Next steps 

You can write a simple Python script to iterate over a set of text excerpts / publications and output a csv file of the full set of interactions that are extracted from these text by the LLM. The example below is very simplified, but you can add additional post-hoc analysis and validations to this, as well as define your `ollama_chat` wrapper as you'd like (as we've done throughout this notebook). This example below can also be found in `src/interaction_extraction_iter.py` in this project root folder. 

```
import requests
import csv
import pandas as pd

# load text from files
with open(path_to_system_ctx, "r") as f:
    system_prompt = f.read().strip()

with open(path_to_example_in, "r") as f:
    example_input = f.read().strip()

with open(path_to_example_out, "r") as f:
    example_output = f.read().strip()

    
def ollama_chat(system_prompt, example_input, example_output, user_prompt):
    """send prompt to llm and return extracted text"""
    try:
        response = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": "llama3.3:70b",
                "stream": False,
                "options": {
                    "num_ctx": 32768
                },
                "messages": [
                    {"role": "system", "content": system_prompt},

                    {"role": "user", "content": example_input},
                    {"role": "assistant", "content": example_output},

                    {"role": "user", "content": user_prompt}
                ]
            }
        )
        return response.json()['message']['content']
    except Exception as e:
        print(f"Error during Llama call: {e}")
        return None

# Helper function to parse pipe delim output into format we can add to table and save as csv.
# This is needed when we have defined pipe delim format explicitly in the system context. 
def parse_csv_output(text):
    """
    parse | delimited CSV text from llm and return list of dicts
    """
    lines = text.strip().splitlines()
    if len(lines) <= 1:
        return []  # Only header or empty

    reader = csv.DictReader(lines, delimiter="|")
    return list(reader)


# Iterate over set of excerpts in path_to_text_excerpts file path. 
all_interactions = []
for i in range(218):
    print(f"Processing paragraph {i+1}/{218}...")
    print(f"path_to_text_excerpts/para_{i}.txt")
    with open(f"path_to_text_excerpts/para_{i}.txt", "r") as f:
        user_prompt = f.read().strip()

    result_text = ollama_chat(system_prompt, example_input, example_output, user_prompt)

    if result_text is None:
        continue  # skip if error

    records = parse_csv_output(result_text)

    if records:
        all_interactions.extend(records)

print(f"Extraction complete. Total interactions found: {len(all_interactions)}")

extr_interactions_df=pd.DataFrame(all_interactions)

extr_interactions_df.to_csv('extracted_interactions.csv')
```